# Evaluate Formulary RAG — Retrieval Quality

`02-Basic_RAG_Formulary.ipynb` proved retrieval changes the answer (with-vs-without-context demo) and now cites its sources, but neither is a substitute for a real evaluation harness: one anecdote isn't evidence the pipeline works, and there was no way to say whether `config['retrieval']['top_k'] = 3` is actually a good setting. This notebook follows the same evaluation approach as `ai-tutor-rag-system/notebooks/06-Evaluate_RAG.ipynb` — build a question set with known-correct answers, then measure **Hit Rate** and **Mean Reciprocal Rank (MRR)** across different `top_k` values — adapted for this project's structured, table-row data instead of prose chunks.

**The one adaptation this requires:** `06-Evaluate_RAG.ipynb` generates its eval questions with an LLM (`generate_question_context_pairs`), because its chunks are free-form prose where a question has to be phrased *around* the content. That's unnecessary work here — and a source of risk, since an LLM can misread a table row that isn't prose. Every formulary row already has the two things a question needs, verbatim in its own metadata: a **drug name** to ask about and a **tier/requirements** answer to check against. So the eval set is **template-generated directly from the fields, deterministically, for free** — no LLM call, no chance of the eval set itself containing a hallucinated question.

**Also reused, not rebuilt:** this notebook doesn't re-parse the PDF. It loads the FAISS cache `02` already built and pulls every row straight out of `vectorstore.docstore` — the eval questions are generated from the *same* documents that are actually indexed, and no re-embedding happens.

**Out of scope for this pass:** answer-quality judging (Faithfulness / Relevancy / Correctness via an LLM-as-judge, as `06-Evaluate_RAG.ipynb` does with `gpt-5`) is a reasonable follow-on — it's an offline eval step, not per-user inference, so a cloud judge wouldn't break this project's local-only serving story. But it's a separate decision (which model judges, at what cost) from retrieval evaluation, so it's left for a later notebook rather than bundled in here.

## 1. Setup

In [1]:
# Same config-driven setup as 02/03 — local embeddings, no API keys.
import yaml
from pathlib import Path

config_path = Path("../../config/config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print(f"✅ Config loaded. Configured top_k: {config['retrieval']['top_k']}")

✅ Config loaded. Configured top_k: 3


## 2. Load the Cached Vector Store (built by `02-Basic_RAG_Formulary.ipynb`)

No PDF parsing and no re-embedding here — if this cache doesn't exist yet, run `02-Basic_RAG_Formulary.ipynb` first so both notebooks evaluate the exact same indexed rows.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=config["embedding"]["model_name"])

MODEL_TAG = config["embedding"]["model_name"].replace("/", "-")
CACHE_DIR = Path(f"../../{config['paths']['processed_data']}/faiss_cache_{MODEL_TAG}_{config['embedding']['dimension']}d")
INDEX_NAME = "formulary_rows"

assert (CACHE_DIR / f"{INDEX_NAME}.faiss").exists(), (
    f"No cache at {CACHE_DIR} — run 02-Basic_RAG_Formulary.ipynb first to build it."
)

vectorstore = FAISS.load_local(
    str(CACHE_DIR), embeddings, index_name=INDEX_NAME, allow_dangerous_deserialization=True
)

# Pull every indexed row straight out of the vector store's own docstore — the
# eval set must be built from exactly what's indexed, not a fresh PDF re-parse
# that could silently drift from it.
all_docs = list(vectorstore.docstore._dict.values())
print(f"✅ Loaded {len(all_docs)} indexed rows from {CACHE_DIR}")

/tmp/ipykernel_28160/3952164510.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/home/bhargav/portfolio-project/healthins-enrollment-assistant-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3349.41it/s]

✅ Loaded 2394 indexed rows from ../../data/processed/NC/2026/formulary/faiss_cache_sentence-transformers-all-MiniLM-L6-v2_384d


## 3. Template-Generated Eval Set (no LLM call)

One question per sampled row, built directly from its metadata. The ground truth for "was this retrieved?" is the `(drug, page)` pair — `page` is included because a drug name alone isn't guaranteed unique across the whole formulary (the same drug can reappear in more than one section).

In [3]:
import json
import random

random.seed(42)

# Retrieval-only eval is cheap here (local embeddings, no LLM in the loop) — bump
# this toward len(all_docs) for a full sweep; kept smaller by default so the notebook
# runs quickly.
EVAL_SAMPLE_SIZE = 200
sampled_docs = random.sample(all_docs, min(EVAL_SAMPLE_SIZE, len(all_docs)))

eval_set = [
    {
        "question": f"What tier and requirements/limits apply to {doc.metadata['drug']}?",
        "expected_drug": doc.metadata["drug"],
        "expected_page": doc.metadata["page"],
    }
    for doc in sampled_docs
]

# Persisted, not just an in-memory list — so the actual 200 questions used for every
# retrieval number in this project are inspectable on disk, and so other notebooks
# (05-Hybrid_Search_Formulary.ipynb) can evaluate against the exact same set instead
# of re-deriving it from a duplicated random-sampling snippet.
EVAL_SET_PATH = Path(f"../../{config['paths']['processed_data']}/eval_set_{EVAL_SAMPLE_SIZE}.json")
with open(EVAL_SET_PATH, "w") as f:
    json.dump(eval_set, f, indent=2)

print(f"✅ Built {len(eval_set)} eval questions from indexed rows (no LLM used)")
print(f"✅ Saved to {EVAL_SET_PATH}")
eval_set[0]

✅ Built 200 eval questions from indexed rows (no LLM used)
✅ Saved to ../../data/processed/NC/2026/formulary/eval_set_200.json


{'question': 'What tier and requirements/limits apply to pemetrexed disodium solr 100mg, 500mg?',
 'expected_drug': 'pemetrexed disodium solr 100mg, 500mg',
 'expected_page': 28}

## 4. Retrieval Evaluation: Hit Rate & MRR vs. `top_k`

Same metrics as `06-Evaluate_RAG.ipynb`, computed directly against FAISS instead of through `RetrieverEvaluator`, so the same eval set works whether the retriever is FAISS, Chroma, or anything else later:

- **Hit Rate** — fraction of questions where the correct row appears anywhere in the top-`k` results.
- **MRR** — average of `1 / rank` of the correct row (0 if it's missing from the top-`k` entirely).

In [4]:
import pandas as pd


def evaluate_retrieval(vectorstore, eval_set, k):
    hits = []
    reciprocal_ranks = []

    for item in eval_set:
        results = vectorstore.similarity_search(item["question"], k=k)
        rank = None
        for i, doc in enumerate(results, start=1):
            if (
                doc.metadata.get("drug") == item["expected_drug"]
                and doc.metadata.get("page") == item["expected_page"]
            ):
                rank = i
                break
        hits.append(1 if rank is not None else 0)
        reciprocal_ranks.append(1 / rank if rank is not None else 0)

    return {
        "top_k": k,
        "hit_rate": sum(hits) / len(hits),
        "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks),
    }


results = [evaluate_retrieval(vectorstore, eval_set, k) for k in [1, 2, 3, 5, 10]]
results_df = pd.DataFrame(results)
results_df

,top_k,hit_rate,mrr
0,1,0.785,0.785000
1,2,0.900,0.842500
2,3,0.935,0.854167
3,5,0.980,0.864417
4,10,0.980,0.864417


In [5]:
configured_k = config["retrieval"]["top_k"]
configured_row = results_df[results_df["top_k"] == configured_k]

if not configured_row.empty:
    print(
        f"Configured top_k={configured_k} -> "
        f"hit_rate={configured_row['hit_rate'].iloc[0]:.3f}, "
        f"mrr={configured_row['mrr'].iloc[0]:.3f}"
    )
else:
    print(f"Configured top_k={configured_k} wasn't in the swept values above — add it to the sweep to compare directly.")

Configured top_k=3 -> hit_rate=0.935, mrr=0.854


## 🔑 Key Takeaways

- **Evaluation methodology ports across data shapes; question generation doesn't.** Hit Rate/MRR are just as meaningful over table rows as over prose chunks — but generating the *questions* with an LLM is a prose-specific workaround for not having a clean answer key. This data already has one.
- **Build the eval set from what's actually indexed, not a fresh re-parse.** Pulling rows out of `vectorstore.docstore` guarantees the eval set and the index agree, and costs nothing extra since the cache already exists.
- **A drug name alone isn't a safe identifier.** Matching on `(drug, page)` avoids false hits/misses from the same drug legitimately appearing in more than one place in the formulary.
- **This gives `config['retrieval']['top_k']` an actual justification.** Before this notebook, `top_k=3` was a plausible-looking default; now it's a number you can compare against the Hit Rate/MRR curve above and change with evidence.
- **The eval set itself is saved, not just used once in memory.** `eval_set_200.json` in the processed-data folder is the actual 200 questions behind every number in this notebook — inspectable on its own, and reused as-is by `05-Hybrid_Search_Formulary.ipynb` so that notebook's comparison is apples-to-apples rather than a fresh, potentially-different sample.
- **Answer-quality judging is a separate, deliberate decision — not bundled in here.** Faithfulness/Correctness eval needs an LLM judge, and picking one (cloud vs. local, cost vs. judgment quality) shouldn't be an incidental side effect of adding retrieval metrics.